# 08 - Refactor e Qualidade do Codigo

Notebook híbrido refatorado para usar diretamente o banco `SQLite` `volve_ml_ready.db`, gerar SQL com um modelo local no `Ollama` e executar testes sem chamar o modelo remoto.

A base representa uma série temporal real de produção de um poço offshore do projeto Volve, no Mar do Norte da Noruega.

## Objetivo
- Abrir diretamente o banco `volve_ml_ready.db` como fonte operacional principal.
- Exibir colunas, `shape`, `describe()` e `head(10)` a partir da tabela no `SQLite`.
- Traduzir perguntas em linguagem natural para SQL usando `Qwen2.5-Coder` local.
- Executar consultas em `SQLite` localmente sobre os dados reais.
- Manter a etapa remota desabilitada para evitar consumo de créditos durante os testes de refactor.

## Ordem de execução

1. Garanta que o `Ollama` esteja instalado e com o serviço iniciado: `ollama serve`.
2. Baixe o modelo local para geração de SQL: `ollama pull qwen2.5-coder:7b-instruct`.
3. Execute a célula de abertura do banco `SQLite`, inspeção do `DataFrame` e montagem do dicionário de dados.
4. Execute a célula de configuração do modelo local e do estado do agente.
5. Execute a célula com os nós do fluxo e compile o grafo.
6. Execute a célula final para rodar o teste de estresse e gerar o relatório próprio deste notebook.

Observação: neste notebook a etapa remota fica desabilitada por padrão para evitar consumo de créditos. Se a sua CPU ficar muito lenta com `qwen2.5-coder:7b-instruct`, troque `LOCAL_SQL_MODEL` para `qwen2.5-coder:3b-instruct`.

## Fonte dos Dados

A fonte operacional deste notebook é o banco `volve_ml_ready.db`, gerado previamente a partir de uma versão tratada da base Volve para analytics e machine learning.

### Proveniência reconhecida

- Base pública associada ao projeto **Volve Field Dataset**.
- Dados liberados pela **Equinor** (antiga **Statoil**).
- Campo **Volve**, offshore da Noruega, no **Mar do Norte**, bloco **15/9**.
- Janela histórica de produção do campo: **2008 a 2016**.
- Unidade de produção associada ao campo: **Mærsk Inspirer**.

### Assinatura típica da base Volve

Essa família de dados é normalmente reconhecida por colunas como:

- `DATEPRD`
- `WELL_BORE_CODE`
- `NPD_WELL_BORE_CODE`
- `NPD_WELL_BORE_NAME`
- `NPD_FIELD_NAME`
- `NPD_FACILITY_NAME`
- `ON_STREAM_HRS`
- `AVG_DOWNHOLE_PRESSURE`
- `AVG_DP_TUBING`
- `AVG_CHOKE_SIZE_P`
- `AVG_WHP_P`
- `AVG_WHT_P`
- `AVG_WELL_BORE_OIL_VOL`
- `AVG_WELL_BORE_GAS_VOL`
- `AVG_WELL_BORE_WAT_VOL`

O banco local usado aqui contém uma **versão transformada e reduzida** dessa base, já adaptada para uso analítico e para tarefas de machine learning. Por isso, parte das colunas regulatórias e operacionais clássicas pode não aparecer mais na tabela final, enquanto outras colunas derivadas foram adicionadas.

### Fontes históricas associadas a essa família de dados

- Portal oficial **Volve Data Village**.
- Base regulatória da antiga **NPD** (*Norwegian Petroleum Directorate*), hoje **NOD** (*Norwegian Offshore Directorate*).
- Espelhos e reempacotamentos públicos em **GitHub** usados por pesquisadores.

### Relevância técnica

O dataset Volve se tornou uma base de referência em Oil & Gas para estudos de:

- *Production Surveillance*
- *Well Performance Analytics*
- *Forecasting*
- *Decline Curve Analysis*
- *Anomaly Detection*
- *Machine Learning para Produção*
- *Digital Oilfield*


### Célula 1: Abertura do SQLite, inspeção do DataFrame e leitura do schema

In [5]:
import os
import re
import sqlite3
import time
from typing import Dict, List, Tuple

import pandas as pd
from IPython.display import display


# -----------------------------------------------------------------------------
# BLOCO 1 - Configuração da fonte de dados
# -----------------------------------------------------------------------------
# Esta célula prepara toda a base de conhecimento estrutural do notebook:
# - localiza o banco SQLite;
# - abre a conexão em modo somente leitura;
# - lê a tabela principal;
# - monta metadados para ajudar o modelo a gerar SQL melhor.
DB_NAME = "volve_ml_ready.db"
TABLE_NAME = "volve_ml_ready"
FEATURE_MEASUREMENT_LOCATION = "Derivada em pipeline analitico / data science"


# Resolve a pasta onde o notebook espera encontrar o banco.
def resolve_notebooks_dir() -> str:
    candidate_dir = os.path.join(os.getcwd(), "notebooks")
    if os.path.isdir(candidate_dir):
        return candidate_dir

    return os.getcwd()


def resolve_database_path(notebooks_dir: str, db_name: str) -> str:
    return os.path.join(notebooks_dir, db_name)


# SQLite aceita conexão via URI. Aqui forçamos o modo somente leitura.
def build_read_only_sqlite_uri(db_path: str) -> str:
    return f"file:{db_path}?mode=ro"


# Se uma conexão global antiga existir, fechamos antes de abrir outra.
# Isso evita conexões duplicadas no ambiente do notebook.
def close_connection_if_present(connection_name: str) -> None:
    connection = globals().get(connection_name)
    if connection is None:
        return

    try:
        connection.close()
    except Exception:
        pass


def ensure_database_exists(db_path: str) -> None:
    if os.path.exists(db_path):
        return

    raise FileNotFoundError(f"Banco SQLite não encontrado: {db_path}")


def open_database_read_only(db_path: str) -> sqlite3.Connection:
    sqlite_uri = build_read_only_sqlite_uri(db_path)
    # uri=True permite usar "file:...?...".
    # check_same_thread=False é útil quando o notebook reusa a conexão em
    # funções diferentes durante a mesma sessão.
    return sqlite3.connect(sqlite_uri, uri=True, check_same_thread=False)


# Carrega a tabela principal como DataFrame.
# Isso é útil tanto para inspeção humana quanto para métricas e relatório.
def load_table_dataframe(
    conn: sqlite3.Connection,
    table_name: str,
    limit: int | None = None,
) -> pd.DataFrame:
    sql = f'SELECT * FROM "{table_name}"'

    if limit is not None:
        sql += f" LIMIT {limit}"

    return pd.read_sql(sql, conn)


def load_schema_dataframe(conn: sqlite3.Connection, table_name: str) -> pd.DataFrame:
    # PRAGMA table_info é uma forma nativa do SQLite de descrever o schema.
    return pd.read_sql(f"PRAGMA table_info({table_name})", conn)


# Transforma o schema em uma string compacta para ser usada dentro do prompt.
def build_schema_text(schema_df: pd.DataFrame) -> str:
    schema_parts: List[str] = []

    for _, row in schema_df.iterrows():
        column_name = str(row["name"])
        column_type = str(row["type"] or "TEXT")
        schema_parts.append(f"{column_name} ({column_type})")

    return ", ".join(schema_parts)


# Exibição exploratória inicial do DataFrame.
# Isso ajuda a confirmar se a base carregou corretamente.
def print_dataframe_overview(df: pd.DataFrame) -> None:
    print(f"[DADOS] Banco SQLite carregado de: {DB_PATH}")
    print(f"[DADOS] Tabela operacional: {TABLE_NAME}")
    print("\n[INSPEÇÃO] Colunas do DataFrame:")
    print(df.columns.tolist())
    print(f"\n[INSPEÇÃO] Shape: {df.shape}")
    print("\n[INSPEÇÃO] Describe:")
    display(df.describe(include="all"))
    print("\n[INSPEÇÃO] Head(10):")
    display(df.head(10))


# -----------------------------------------------------------------------------
# BLOCO 2 - Metadados das colunas
# -----------------------------------------------------------------------------
# Este dicionário descreve colunas "base", isto é, colunas originais e mais
# diretamente ligadas ao dado operacional.
BASE_COLUMN_METADATA = {
    "DATEPRD": {
        "descricao": "Data da producao/operacao diaria.",
        "tipo_analitico": "datetime",
        "unidade": "data",
        "origem": "Sistema operacional / historian",
        "local_medicao": "Centro de supervisao / banco operacional",
        "classe": "coluna original do dataset Volve",
    },
    "WELL_TYPE": {
        "descricao": "Tipo do poco, por exemplo produtor ou injetor.",
        "tipo_analitico": "string",
        "unidade": "N/A",
        "origem": "Engenharia de producao",
        "local_medicao": "Configuracao operacional do poco",
        "classe": "coluna original do dataset Volve",
    },
    "ON_STREAM_HRS": {
        "descricao": "Quantidade de horas em operacao/produzindo no dia.",
        "tipo_analitico": "float",
        "unidade": "horas",
        "origem": "Sistema supervisorio / producao",
        "local_medicao": "Status operacional do poco",
        "classe": "coluna original do dataset Volve",
    },
    "AVG_DOWNHOLE_PRESSURE": {
        "descricao": "Pressao media no fundo do poco.",
        "tipo_analitico": "float",
        "unidade": "bar(a)",
        "origem": "Gauge de fundo / sensor downhole",
        "local_medicao": "Fundo do poco / proximo da zona produtora",
        "classe": "coluna original do dataset Volve",
    },
    "AVG_DOWNHOLE_TEMPERATURE": {
        "descricao": "Temperatura media no fundo do poco.",
        "tipo_analitico": "float",
        "unidade": "graus C",
        "origem": "Sensor downhole",
        "local_medicao": "Fundo do poco / tubing inferior",
        "classe": "coluna original do dataset Volve",
    },
    "AVG_DP_TUBING": {
        "descricao": "Delta de pressao medio no tubing.",
        "tipo_analitico": "float",
        "unidade": "bar",
        "origem": "Sensores de pressao no tubing",
        "local_medicao": "Interior do tubing de producao",
        "classe": "coluna original do dataset Volve",
    },
    "AVG_CHOKE_SIZE_P": {
        "descricao": "Abertura media do choke em superficie.",
        "tipo_analitico": "float",
        "unidade": "%",
        "origem": "Atuador/sensor do choke",
        "local_medicao": "Choke na arvore de natal / superficie",
        "classe": "coluna original do dataset Volve",
    },
    "AVG_WHP_P": {
        "descricao": "Pressao media na cabeca do poco.",
        "tipo_analitico": "float",
        "unidade": "bar",
        "origem": "Sensor wellhead",
        "local_medicao": "Cabeca do poco / arvore de natal",
        "classe": "coluna original do dataset Volve",
    },
    "AVG_WHT_P": {
        "descricao": "Temperatura media na cabeca do poco.",
        "tipo_analitico": "float",
        "unidade": "graus C",
        "origem": "Sensor wellhead",
        "local_medicao": "Cabeca do poco / arvore de natal",
        "classe": "coluna original do dataset Volve",
    },
    "BORE_OIL_VOL": {
        "descricao": "Volume diario de oleo produzido.",
        "tipo_analitico": "float",
        "unidade": "Sm3/d",
        "origem": "Medidor multifasico / teste de producao",
        "local_medicao": "Linha de producao do poco",
        "classe": "coluna original do dataset Volve",
    },
    "BORE_WAT_VOL": {
        "descricao": "Volume diario de agua produzida.",
        "tipo_analitico": "float",
        "unidade": "Sm3/d",
        "origem": "Medidor multifasico / separador",
        "local_medicao": "Linha de producao / separador",
        "classe": "coluna original do dataset Volve",
    },
}


# Este dicionário resume a família de features derivadas.
# Ele serve como base para gerar descrições automáticas de colunas de ML.
FEATURE_BASE_INFO = {
    "oil": {
        "descricao_base": "serie de oleo derivada de BORE_OIL_VOL",
        "unidade": "Sm3/d",
        "origem": "Feature engineering a partir de BORE_OIL_VOL",
    },
    "water": {
        "descricao_base": "serie de agua derivada de BORE_WAT_VOL",
        "unidade": "Sm3/d",
        "origem": "Feature engineering a partir de BORE_WAT_VOL",
    },
    "gas": {
        "descricao_base": "serie de gas derivada da familia BORE_GAS_VOL do dataset Volve original",
        "unidade": "Sm3/d",
        "origem": "Feature engineering a partir da serie de gas do dataset fonte",
    },
}


# Cria um bloco padrão de metadados para uma feature derivada.
def build_feature_metadata(
    descricao: str,
    tipo_analitico: str,
    unidade: str,
    origem: str,
    classe: str,
) -> Dict[str, str]:
    return {
        "descricao": descricao,
        "tipo_analitico": tipo_analitico,
        "unidade": unidade,
        "origem": origem,
        "local_medicao": FEATURE_MEASUREMENT_LOCATION,
        "classe": classe,
    }


# Variação do helper anterior: parte de uma "família" (oil/water/gas)
# e apenas sobrescreve unidade/origem quando necessário.
def build_series_feature_metadata(
    base_info: Dict[str, str],
    descricao: str,
    tipo_analitico: str,
    classe: str,
    unidade: str | None = None,
    origem: str | None = None,
) -> Dict[str, str]:
    final_unit = unidade if unidade is not None else base_info["unidade"]
    final_origin = origem if origem is not None else base_info["origem"]
    return build_feature_metadata(
        descricao=descricao,
        tipo_analitico=tipo_analitico,
        unidade=final_unit,
        origem=final_origin,
        classe=classe,
    )


# Descobre a qual família de feature a coluna pertence.
def get_feature_base_info(column_name: str) -> Tuple[str, Dict[str, str] | None]:
    lower_name = column_name.lower()

    for prefix, base_info in FEATURE_BASE_INFO.items():
        if lower_name.startswith(f"{prefix}_"):
            return prefix, base_info

    return "", None


# Algumas colunas não seguem exatamente um padrão repetitivo simples,
# então tratamos seus casos especiais aqui.
def infer_special_feature_metadata(lower_name: str) -> Dict[str, str] | None:
    oil_base = FEATURE_BASE_INFO["oil"]
    water_base = FEATURE_BASE_INFO["water"]

    if lower_name == "oil_roll_30":
        return build_series_feature_metadata(
            oil_base,
            "Feature rolling de 30 dias aplicada a serie de oleo.",
            "feature derivada rolling",
            "feature derivada - rolling",
        )

    if lower_name == "oil_expanding_mean":
        return build_series_feature_metadata(
            oil_base,
            "Media expansiva acumulada da serie de oleo ao longo do tempo.",
            "feature derivada expanding mean",
            "feature derivada - expanding mean",
        )

    if lower_name == "oil_expanding_std":
        return build_series_feature_metadata(
            oil_base,
            "Desvio padrao expansivo acumulado da serie de oleo ao longo do tempo.",
            "feature derivada expanding std",
            "feature derivada - expanding std",
        )

    if lower_name == "water_cumulative":
        return build_series_feature_metadata(
            water_base,
            "Acumulado historico da serie de agua produzida ao longo do periodo.",
            "feature derivada cumulativa",
            "feature derivada - cumulativa",
            unidade="Sm3 acumulado",
        )

    if lower_name == "oil_acceleration":
        return build_series_feature_metadata(
            oil_base,
            "Indicador derivado de aceleracao da dinamica da serie de oleo.",
            "feature derivada de segunda ordem",
            "feature derivada - aceleracao",
            unidade="unidade derivada do pipeline analitico",
        )

    if lower_name in {"oil_trend_strength", "water_trend_strength"}:
        family = "oleo" if lower_name.startswith("oil") else "agua"
        return build_feature_metadata(
            descricao=f"Indicador derivado de forca de tendencia da serie de {family}.",
            tipo_analitico="feature derivada de tendencia",
            unidade="unidade derivada do pipeline analitico",
            origem="Feature engineering temporal",
            classe="feature derivada - trend strength",
        )

    if lower_name in {"oil_vs_trend", "water_vs_trend"}:
        family = "oleo" if lower_name.startswith("oil") else "agua"
        return build_feature_metadata(
            descricao=f"Razao entre o valor corrente e a tendencia estimada da serie de {family}.",
            tipo_analitico="feature derivada de comparacao com tendencia",
            unidade="adimensional",
            origem="Feature engineering temporal",
            classe="feature derivada - vs trend",
        )

    if lower_name == "oil_volatility_index":
        return build_feature_metadata(
            descricao="Indice derivado de volatilidade da serie de oleo.",
            tipo_analitico="feature derivada de volatilidade",
            unidade="adimensional",
            origem="Feature engineering temporal",
            classe="feature derivada - volatility index",
        )

    if lower_name == "oil_momentum_30d":
        return build_series_feature_metadata(
            oil_base,
            "Momentum de 30 dias da serie de oleo, comparando o valor corrente com a referencia de 30 dias antes.",
            "feature derivada de momentum",
            "feature derivada - momentum",
        )

    if lower_name == "oil_roc_30d":
        return build_series_feature_metadata(
            oil_base,
            "Rate of change de 30 dias da serie de oleo.",
            "feature derivada de taxa de variacao",
            "feature derivada - rate of change",
            unidade="fracao ou % conforme convencao do pipeline",
        )

    if lower_name == "oil_zscore_30":
        return build_series_feature_metadata(
            oil_base,
            "Z-score de 30 dias da serie de oleo.",
            "feature derivada de padronizacao",
            "feature derivada - zscore",
            unidade="adimensional",
        )

    return None


# Caso nenhuma regra específica sirva, caímos neste metadado genérico.
def build_default_feature_metadata() -> Dict[str, str]:
    return {
        "descricao": "Coluna derivada presente na base tratada para analytics e machine learning.",
        "tipo_analitico": "feature derivada",
        "unidade": "dependente da transformacao",
        "origem": "Pipeline analitico / feature engineering",
        "local_medicao": "Derivada em ambiente analitico",
        "classe": "feature derivada",
    }


# Esta função é a "inteligência semântica local" das features derivadas.
# Ela tenta explicar automaticamente o significado de cada coluna calculada.
def infer_feature_metadata(column_name: str) -> Dict[str, str]:
    lower_name = column_name.lower()
    _, base_info = get_feature_base_info(lower_name)

    if base_info is not None:
        if lower_name.startswith(("oil_lag_", "water_lag_", "gas_lag_")):
            days = lower_name.split("_")[-1]
            return build_series_feature_metadata(
                base_info,
                f"Valor defasado em {days} dias da {base_info['descricao_base']}.",
                "feature derivada temporal",
                "feature derivada - lag temporal",
            )

        if lower_name.startswith(("oil_roll_mean_", "water_roll_mean_", "gas_roll_mean_")):
            window = lower_name.split("_")[-1]
            return build_series_feature_metadata(
                base_info,
                f"Media movel de {window} dias da {base_info['descricao_base']}.",
                "feature derivada rolling mean",
                "feature derivada - rolling mean",
            )

        if lower_name.startswith(("oil_roll_std_", "water_roll_std_", "gas_roll_std_")):
            window = lower_name.split("_")[-1]
            return build_series_feature_metadata(
                base_info,
                f"Desvio padrao movel de {window} dias da {base_info['descricao_base']}.",
                "feature derivada rolling std",
                "feature derivada - rolling std",
            )

        if lower_name.startswith(("oil_delta_", "water_delta_", "gas_delta_")):
            days = lower_name.split("_")[-1].replace("d", "")
            return build_series_feature_metadata(
                base_info,
                f"Diferenca da {base_info['descricao_base']} em relacao a {days} dias antes.",
                "feature derivada delta",
                "feature derivada - delta temporal",
            )

        if lower_name.startswith(
            ("oil_pct_change_", "water_pct_change_", "gas_pct_change_")
        ):
            days = lower_name.split("_")[-1].replace("d", "")
            return build_series_feature_metadata(
                base_info,
                f"Variacao percentual da {base_info['descricao_base']} em relacao a {days} dias antes.",
                "feature derivada percentual",
                "feature derivada - pct change",
                unidade="%",
            )

    special_metadata = infer_special_feature_metadata(lower_name)
    if special_metadata is not None:
        return special_metadata

    return build_default_feature_metadata()


# Primeiro tentamos o catálogo explícito das colunas base.
# Se não existir, inferimos a descrição automaticamente.
def get_column_metadata(column_name: str, column_type: str) -> Dict[str, str]:
    if column_name in BASE_COLUMN_METADATA:
        return BASE_COLUMN_METADATA[column_name]

    return infer_feature_metadata(column_name)


# -----------------------------------------------------------------------------
# BLOCO 3 - Utilitários de parsing e normalização de SQL
# -----------------------------------------------------------------------------
# Estes helpers existem porque o LLM pode gerar SQL quase correto, mas com
# alias desnecessário ou com expressões que precisam de limpeza antes de uso.
def dedupe_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    ordered: List[str] = []

    for item in items:
        if item in seen:
            continue

        seen.add(item)
        ordered.append(item)

    return ordered


# Remove aspas, crases e prefixos de tabela para deixar o nome da coluna limpo.
def normalize_sql_identifier(identifier: str) -> str:
    normalized = str(identifier).strip()
    normalized = normalized.strip('"').strip("`").strip()

    if "." in normalized:
        normalized = normalized.split(".")[-1]

    return normalized


# Extrai apenas o trecho SELECT ... FROM para poder analisar a lista de colunas.
def extract_select_clause(sql: str) -> str:
    match = re.search(r"(?is)^\s*select\s+(.*?)\s+from\s", sql.strip())
    if match is None:
        return ""

    return match.group(1).strip()


# Divide a lista de itens do SELECT respeitando parênteses.
# Isso evita quebrar expressões como AVG(...), MAX(...), etc.
def split_sql_select_items(select_clause: str) -> List[str]:
    items: List[str] = []
    current_chars: List[str] = []
    depth = 0

    for char in select_clause:
        if char == "(":
            depth += 1
        elif char == ")":
            depth = max(depth - 1, 0)

        if char == "," and depth == 0:
            item = "".join(current_chars).strip()
            if item:
                items.append(item)
            current_chars = []
            continue

        current_chars.append(char)

    final_item = "".join(current_chars).strip()
    if final_item:
        items.append(final_item)

    return items


# Tenta descobrir qual coluna real do schema aparece dentro de uma expressão SQL.
def detect_source_column_from_expression(expression: str) -> str:
    normalized_expression = str(expression).upper()
    matches: List[str] = []

    for column_name in BASE_SCHEMA_COLUMNS:
        pattern = rf"(?<![A-Z0-9_]){re.escape(column_name.upper())}(?![A-Z0-9_])"
        if re.search(pattern, normalized_expression):
            matches.append(column_name)

    unique_matches = dedupe_preserve_order(matches)
    if len(unique_matches) == 1:
        return unique_matches[0]

    return ""


# Se o modelo escreveu "AVG(x) AS media_x", separamos:
# - expressão;
# - alias;
# - coluna de origem.
def parse_sql_select_item(item: str) -> Dict[str, str]:
    stripped_item = item.strip()
    alias_pattern = r'(?is)^(.*?)\s+AS\s+("?[A-Za-z_][A-Za-z0-9_]*"?)\s*$'
    alias_match = re.match(alias_pattern, stripped_item)

    expression = stripped_item
    alias = ""

    if alias_match is not None:
        expression = alias_match.group(1).strip()
        alias = normalize_sql_identifier(alias_match.group(2))

    source_column = detect_source_column_from_expression(expression)
    return {
        "raw_item": stripped_item,
        "expression": expression,
        "alias": alias,
        "source_column": source_column,
    }


# Remove alias desnecessário em colunas diretas para manter a resposta mais
# alinhada ao schema original.
def normalize_generated_sql(sql: str) -> str:
    cleaned_sql = str(sql).strip()
    select_clause = extract_select_clause(cleaned_sql)
    if not select_clause:
        return cleaned_sql

    normalized_items: List[str] = []

    for item in split_sql_select_items(select_clause):
        parsed_item = parse_sql_select_item(item)
        if parsed_item["alias"] and parsed_item["source_column"]:
            normalized_items.append(parsed_item["expression"])
            continue

        normalized_items.append(parsed_item["raw_item"])

    normalized_select_clause = ", ".join(normalized_items)
    select_pattern = r"(?is)^(\s*select\s+)(.*?)(\s+from\s)"

    def replace_select_clause(match: re.Match[str]) -> str:
        prefix = match.group(1)
        suffix = match.group(3)
        return f"{prefix}{normalized_select_clause}{suffix}"

    return re.sub(select_pattern, replace_select_clause, cleaned_sql, count=1)


# -----------------------------------------------------------------------------
# BLOCO 4 - Heurísticas simples para escolher contexto
# -----------------------------------------------------------------------------
# Estas pistas ajudam o notebook a selecionar colunas mais relevantes para
# a pergunta antes mesmo de chamar o modelo local.
QUESTION_KEYWORD_HINTS = {
    "oleo": ["BORE_OIL_VOL"],
    "óleo": ["BORE_OIL_VOL"],
    "oil": ["BORE_OIL_VOL"],
    "agua": ["BORE_WAT_VOL"],
    "água": ["BORE_WAT_VOL"],
    "water": ["BORE_WAT_VOL"],
    "pressao": ["AVG_DOWNHOLE_PRESSURE"],
    "pressão": ["AVG_DOWNHOLE_PRESSURE"],
    "downhole": ["AVG_DOWNHOLE_PRESSURE", "AVG_DOWNHOLE_TEMPERATURE"],
    "temperatura": ["AVG_DOWNHOLE_TEMPERATURE"],
    "temperature": ["AVG_DOWNHOLE_TEMPERATURE"],
    "stream": ["ON_STREAM_HRS"],
    "hora": ["ON_STREAM_HRS"],
    "horas": ["ON_STREAM_HRS"],
}


LOCAL_SQL_FALLBACK_COLUMNS = [
    "DATEPRD",
    "ON_STREAM_HRS",
    "AVG_DOWNHOLE_PRESSURE",
    "AVG_DOWNHOLE_TEMPERATURE",
    "BORE_OIL_VOL",
    "BORE_WAT_VOL",
    "WELL_TYPE",
]


# Constrói uma linha textual rica em contexto para uma coluna específica.
def build_column_context_line(column_name: str, column_type: str) -> str:
    metadata = get_column_metadata(column_name, column_type)
    line_parts = [
        f"- {column_name}",
        f"sql_type={column_type}",
        f"classe={metadata['classe']}",
        f"descricao={metadata['descricao']}",
        f"tipo_analitico={metadata['tipo_analitico']}",
        f"unidade={metadata['unidade']}",
        f"origem={metadata['origem']}",
        f"local_medicao={metadata['local_medicao']}",
    ]
    return " | ".join(line_parts)


# Gera um dicionário de dados textual completo.
# Esse texto depois é usado tanto no relatório quanto como contexto semântico.
def build_data_dictionary(
    schema_df: pd.DataFrame,
) -> Tuple[str, Dict[str, str]]:
    lines = [
        f"TABELA: {TABLE_NAME}",
        "CONTEXTO: Serie temporal real de producao de um poco offshore do projeto Volve, no Mar do Norte da Noruega, carregada a partir do banco SQLite operacional do projeto.",
        "",
        "OBSERVACOES GERAIS:",
        "- Esta tabela contem apenas as colunas disponiveis na tabela operacional volve_ml_ready.",
        "- Algumas colunas classicas do dataset bruto Volve nao estao presentes nesta versao tratada.",
        "- Colunas prefixadas com oil_, water_ e gas_ sao features derivadas usadas para analytics e machine learning.",
        "- Para SQL, use apenas os nomes de coluna listados abaixo exatamente como aparecem.",
        "",
        "COLUNAS DISPONIVEIS NESTA TABELA:",
    ]
    context_lines_by_column: Dict[str, str] = {}

    for _, row in schema_df.iterrows():
        column_name = str(row["name"])
        column_type = str(row["type"] or "TEXT")
        line = build_column_context_line(column_name, column_type)
        context_lines_by_column[column_name] = line
        lines.append(line)

    return "\n".join(lines), context_lines_by_column


# Seleciona colunas potencialmente relevantes para a pergunta.
# O objetivo é reduzir contexto inútil e destacar o que mais importa.
def select_relevant_columns_for_question(question: str) -> List[str]:
    normalized_question = str(question).lower()
    relevant_columns: List[str] = []

    for column_name in COLUMN_CONTEXT_LINE_BY_NAME:
        if column_name.lower() in normalized_question:
            relevant_columns.append(column_name)

    for keyword, columns in QUESTION_KEYWORD_HINTS.items():
        if keyword not in normalized_question:
            continue

        for column_name in columns:
            if column_name in COLUMN_CONTEXT_LINE_BY_NAME:
                relevant_columns.append(column_name)

    if any(token in normalized_question for token in ["data", "date", "dia", "temporal"]):
        if "DATEPRD" in COLUMN_CONTEXT_LINE_BY_NAME:
            relevant_columns.append("DATEPRD")

    relevant_columns = dedupe_preserve_order(relevant_columns)
    if relevant_columns:
        return relevant_columns

    fallback_columns: List[str] = []
    for column_name in LOCAL_SQL_FALLBACK_COLUMNS:
        if column_name in COLUMN_CONTEXT_LINE_BY_NAME:
            fallback_columns.append(column_name)

    return fallback_columns


# Detecta colunas explicitamente citadas pelo usuário na pergunta.
def extract_explicit_schema_columns_from_question(question: str) -> List[str]:
    normalized_question = str(question).lower()
    explicit_columns: List[str] = []

    for column_name in SCHEMA_COLUMN_NAMES:
        if column_name.lower() in normalized_question:
            explicit_columns.append(column_name)

    return explicit_columns


# Tokenização simples para facilitar regras baseadas em palavras.
def normalize_question_words(question: str) -> List[str]:
    cleaned_question = re.sub(r"[^\w]+", " ", str(question).lower(), flags=re.UNICODE)
    return [word for word in cleaned_question.split() if word]


# Fast path = tradução por regra, sem chamar o LLM.
# É útil para perguntas objetivas como máximo, mínimo ou média de uma coluna.
def try_build_rule_based_sql(question: str) -> str:
    explicit_columns = extract_explicit_schema_columns_from_question(question)
    explicit_columns = [name for name in explicit_columns if name != "DATEPRD"]

    if len(explicit_columns) != 1:
        return ""

    column_name = explicit_columns[0]
    normalized_words = set(normalize_question_words(question))

    avg_tokens = {"média", "media"}
    max_tokens = {"maior", "máxima", "maxima", "máximo", "maximo", "pico"}
    min_tokens = {"menor", "mínima", "minima", "mínimo", "minimo"}

    if normalized_words & avg_tokens:
        alias_name = f"avg_{column_name.lower()}"
        return f"SELECT AVG({column_name}) AS {alias_name} FROM {TABLE_NAME}"

    if normalized_words & max_tokens:
        return (
            f"SELECT DATEPRD, {column_name} "
            f"FROM {TABLE_NAME} "
            f"ORDER BY {column_name} DESC LIMIT 1"
        )

    if normalized_words & min_tokens:
        return (
            f"SELECT DATEPRD, {column_name} "
            f"FROM {TABLE_NAME} "
            f"ORDER BY {column_name} ASC LIMIT 1"
        )

    return ""


# Monta um contexto curto para o prompt local.
# A ideia é não enviar o dicionário inteiro se apenas algumas colunas já bastam.
def build_local_sql_context(question: str) -> str:
    relevant_columns = select_relevant_columns_for_question(question)
    lines = [
        "CONTEXTO ENXUTO PARA GERAR SQL:",
        "- Use o schema completo para nomes e tipos de coluna.",
        "- Use somente as linhas abaixo como apoio semantico das colunas mais provaveis desta pergunta.",
        "- Colunas prefixadas com oil_, water_ e gas_ sao features derivadas. Interprete o sufixo para distinguir lag, media movel, desvio, delta, percentual, cumulativo ou tendencia.",
        "",
        "COLUNAS MAIS RELEVANTES PARA ESTA PERGUNTA:",
    ]

    for column_name in relevant_columns:
        lines.append(COLUMN_CONTEXT_LINE_BY_NAME[column_name])

    return "\n".join(lines)


# Depois da execução SQL, reconstruímos o significado das colunas devolvidas.
# Isso ajuda muito na fase de explicação e debug.
def build_query_column_context(sql: str, columns: List[str]) -> str:
    lines = ["COLUNAS RETORNADAS PELA CONSULTA:"]
    select_clause = extract_select_clause(sql)
    parsed_items: List[Dict[str, str]] = []

    for item in split_sql_select_items(select_clause):
        parsed_items.append(parse_sql_select_item(item))

    for index, column_name in enumerate(columns):
        parsed_item = parsed_items[index] if index < len(parsed_items) else {}
        source_column = parsed_item.get("source_column", "")
        metadata_column_name = source_column or column_name
        metadata = get_column_metadata(metadata_column_name, "RESULT")

        line_parts = [f"- {column_name}"]
        if source_column and source_column != column_name:
            line_parts.append(f"coluna_origem={source_column}")

        line_parts.extend(
            [
                f"classe={metadata['classe']}",
                f"descricao={metadata['descricao']}",
                f"tipo_analitico={metadata['tipo_analitico']}",
                f"unidade={metadata['unidade']}",
            ]
        )
        lines.append(" | ".join(line_parts))

    return "\n".join(lines)


# Texto institucional e técnico da fonte de dados.
DATA_SOURCE_TEXT = """
BASE OPERACIONAL ATUAL: volve_ml_ready.db
TABELA OPERACIONAL ATUAL: volve_ml_ready
PROVENIÊNCIA: banco SQLite gerado previamente a partir de uma versão tratada para analytics e machine learning derivada do Volve Field Dataset.
LIBERAÇÃO PÚBLICA ORIGINAL: Equinor (antiga Statoil).
CONTEXTO OPERACIONAL:
- Campo Volve, offshore da Noruega, no Mar do Norte, bloco 15/9.
- Produção histórica do campo entre 2008 e 2016.
- Unidade de produção associada: Mærsk Inspirer.
OBSERVAÇÕES:
- O banco local usado neste notebook foi gerado previamente e agora é a fonte oficial desta execução.
- A versão original de trabalho foi tratada para analytics e ML antes de chegar a este banco SQLite.
- Parte das colunas clássicas do dataset Volve pode ter sido transformada, removida ou enriquecida durante a preparação.
- A família de dados Volve ficou conhecida por disponibilizar séries reais de produção e variáveis operacionais para pesquisa e indústria.
FONTES HISTÓRICAS ASSOCIADAS À FAMÍLIA DE DADOS:
- Volve Data Village.
- Base regulatória NPD, atualmente NOD.
- Espelhos públicos em GitHub usados por pesquisadores.
""".strip()


# -----------------------------------------------------------------------------
# BLOCO 5 - Bootstrap da célula
# -----------------------------------------------------------------------------
# A partir daqui saímos da definição de funções e executamos o setup real.
NOTEBOOKS_DIR = resolve_notebooks_dir()
DB_PATH = resolve_database_path(NOTEBOOKS_DIR, DB_NAME)

close_connection_if_present("conn")
ensure_database_exists(DB_PATH)

# Conexão inicial usada para carregar dados e schema.
setup_conn = open_database_read_only(DB_PATH)
source_df = load_table_dataframe(setup_conn, TABLE_NAME)
schema_df = load_schema_dataframe(setup_conn, TABLE_NAME)
schema_text = build_schema_text(schema_df)

# Lista simples com os nomes das colunas do schema.
SCHEMA_COLUMN_NAMES: List[str] = []
for column_name in schema_df["name"].tolist():
    SCHEMA_COLUMN_NAMES.append(str(column_name))

# Estas duas estruturas serão muito usadas pelo restante do notebook:
# - BASE_SCHEMA_COLUMNS: facilita detecção de colunas dentro do SQL;
# - COLUMN_CONTEXT_LINE_BY_NAME: facilita recuperar contexto textual por coluna.
BASE_SCHEMA_COLUMNS = set(SCHEMA_COLUMN_NAMES)
DATA_DICTIONARY, COLUMN_CONTEXT_LINE_BY_NAME = build_data_dictionary(schema_df)

# Visualização inicial dos dados.
print_dataframe_overview(source_df)

# Fechamos a conexão de setup e abrimos a conexão operacional principal.
setup_conn.close()
conn = open_database_read_only(DB_PATH)

print("\n[GOVERNANÇA] Banco de dados inicializado em modo read-only.")
display(schema_df)

# Preview final direto do banco para confirmar que a conexão operacional está OK.
db_preview_df = load_table_dataframe(conn, TABLE_NAME, limit=10)
print("\n[VALIDAÇÃO] Leitura dos 10 primeiros registros a partir do SQLite:")
display(db_preview_df)


[DADOS] Banco SQLite carregado de: /home/wolf/Documentos/lab-artificial-inteligence/notebooks/volve_ml_ready.db
[DADOS] Tabela operacional: volve_ml_ready

[INSPEÇÃO] Colunas do DataFrame:
['DATEPRD', 'ON_STREAM_HRS', 'AVG_DOWNHOLE_PRESSURE', 'AVG_DOWNHOLE_TEMPERATURE', 'AVG_DP_TUBING', 'AVG_CHOKE_SIZE_P', 'AVG_WHP_P', 'AVG_WHT_P', 'BORE_OIL_VOL', 'BORE_WAT_VOL', 'WELL_TYPE', 'oil_roll_30', 'oil_lag_1', 'water_lag_1', 'oil_lag_3', 'water_lag_3', 'oil_lag_7', 'water_lag_7', 'oil_lag_14', 'water_lag_14', 'oil_lag_30', 'gas_lag_30', 'water_lag_30', 'oil_roll_mean_3', 'oil_roll_mean_7', 'water_roll_mean_7', 'oil_roll_mean_14', 'water_roll_mean_30', 'oil_roll_std_7', 'water_roll_std_7', 'oil_roll_std_14', 'water_roll_std_14', 'oil_roll_std_30', 'water_roll_std_30', 'oil_delta_1d', 'water_delta_1d', 'oil_delta_3d', 'water_delta_3d', 'oil_delta_7d', 'water_delta_7d', 'oil_pct_change_1d', 'water_pct_change_1d', 'oil_pct_change_7d', 'water_pct_change_7d', 'oil_pct_change_14d', 'water_pct_change

,DATEPRD,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_CHOKE_SIZE_P,AVG_WHP_P,AVG_WHT_P,BORE_OIL_VOL,BORE_WAT_VOL,...,water_cumulative,oil_acceleration,oil_trend_strength,water_trend_strength,oil_vs_trend,water_vs_trend,oil_volatility_index,oil_momentum_30d,oil_roc_30d,oil_zscore_30
count,125,125.000000,125.000000,125.000000,125.000000,125.000000,125.000000,125.000000,125.000000,125.00000,...,125.000000,125.000000,125.000000,125.000000,125.000000,125.000000,125.000000,125.00000,125.000000,125.000000
unique,125,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,2014-06-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,23.648800,217.614074,108.207550,172.940248,48.577485,44.673826,56.494484,428.729760,235.05424,...,17311.774800,-5.497360,-28.274622,24.682399,0.959332,1.226105,0.138246,-85.93680,-0.071723,-0.653451
std,NaN,2.307786,3.404902,0.157091,11.141340,4.271622,9.224183,4.048699,161.042726,125.96291,...,16672.230599,117.268766,60.725764,51.040085,0.337634,0.494197,0.129095,143.82898,0.597412,1.163929
min,NaN,0.991660,209.822862,107.447354,154.685158,15.614031,28.380119,44.267469,0.000000,0.00000,...,1252.950000,-448.650000,-158.939190,-85.579762,0.000000,0.000000,0.008125,-705.74000,-1.000000,-5.113725
25%,NaN,24.000000,215.154616,108.134254,162.789418,47.607174,36.665793,54.294324,296.200000,94.65000,...,3482.520000,-13.500000,-74.273381,-6.772524,0.840121,0.969086,0.045412,-169.01000,-0.279748,-1.155068
50%,NaN,24.000000,216.044918,108.202384,171.445014,48.698561,42.617301,56.696216,378.380000,247.16000,...,10782.010000,-1.680000,-21.787857,18.386571,0.939541,1.071721,0.107189,-52.27000,-0.119289,-0.594634
75%,NaN,24.000000,220.275472,108.342534,183.488217,50.800792,52.663308,58.737828,518.320000,323.66000,...,29999.660000,7.550000,7.547429,35.614238,1.061093,1.337841,0.201331,-23.09000,-0.044570,0.253275



[INSPEÇÃO] Head(10):


,DATEPRD,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_CHOKE_SIZE_P,AVG_WHP_P,AVG_WHT_P,BORE_OIL_VOL,BORE_WAT_VOL,...,water_cumulative,oil_acceleration,oil_trend_strength,water_trend_strength,oil_vs_trend,water_vs_trend,oil_volatility_index,oil_momentum_30d,oil_roc_30d,oil_zscore_30
0,2014-06-06,24.0,220.749795,107.986806,157.674225,47.202699,63.075569,56.534234,690.38,92.18,...,1252.95,42.60,-53.398714,36.586429,0.954443,2.207111,0.101595,-234.93,-0.253893,-0.251521
1,2014-06-07,24.0,220.640141,107.994109,157.278305,47.194107,63.361836,56.615421,694.30,92.07,...,1345.02,-9.17,-49.170048,35.386000,0.969673,2.053575,0.099854,-219.56,-0.240256,-0.172277
2,2014-06-08,24.0,220.504050,108.002713,157.184594,47.152024,63.319457,52.051159,690.70,91.25,...,1436.27,-7.52,-71.087190,33.940048,0.942106,1.905979,0.097347,513.90,2.906674,-0.568233
3,2014-06-09,19.5,221.450514,107.757260,159.113239,38.978911,62.337275,53.559224,238.45,28.15,...,1464.42,-448.65,-79.188238,30.421714,0.336025,0.576679,0.208615,-705.74,-0.747455,-4.317439
4,2014-06-10,24.0,220.851692,107.990754,158.074501,46.512532,62.777191,54.486153,647.11,84.88,...,1549.30,860.91,-83.024429,28.189524,0.919526,1.643581,0.208276,-176.30,-0.214110,-0.526682
5,2014-06-11,24.0,218.414712,108.065274,156.685164,47.815208,61.729549,54.295347,745.92,94.23,...,1643.53,-309.85,-76.666190,27.119952,1.061093,1.720017,0.210288,-23.09,-0.030026,0.400897
6,2014-06-12,24.0,215.496859,108.139361,155.559215,48.727774,59.937644,61.561027,804.85,101.88,...,1745.41,-39.88,-60.578333,25.339667,1.141456,1.751107,0.216511,64.05,0.086461,0.918978
7,2014-06-13,24.0,214.823228,108.135199,154.685158,48.263917,60.138070,62.327179,818.89,56.75,...,1802.16,-44.89,-45.741429,18.386571,1.155596,0.944700,0.222879,105.65,0.148127,0.997787
8,2014-06-14,24.0,215.832541,108.104783,155.597785,48.005914,60.234755,59.584407,781.21,54.32,...,1856.48,-51.72,-34.571714,11.183048,1.100488,0.877790,0.225316,37.38,0.050253,0.641931
9,2014-06-15,24.0,214.877321,108.113804,154.957708,48.040325,59.919612,60.774161,786.16,54.55,...,1911.03,42.63,-22.603571,4.121857,1.104863,0.856344,0.227629,50.07,0.068022,0.666773



[GOVERNANÇA] Banco de dados inicializado em modo read-only.


,cid,name,type,notnull,dflt_value,pk
0,0,DATEPRD,TEXT,0,None,0
1,1,ON_STREAM_HRS,REAL,0,None,0
2,2,AVG_DOWNHOLE_PRESSURE,REAL,0,None,0
3,3,AVG_DOWNHOLE_TEMPERATURE,REAL,0,None,0
4,4,AVG_DP_TUBING,REAL,0,None,0
5,5,AVG_CHOKE_SIZE_P,REAL,0,None,0
6,6,AVG_WHP_P,REAL,0,None,0
7,7,AVG_WHT_P,REAL,0,None,0
8,8,BORE_OIL_VOL,REAL,0,None,0
9,9,BORE_WAT_VOL,REAL,0,None,0



[VALIDAÇÃO] Leitura dos 10 primeiros registros a partir do SQLite:


,DATEPRD,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_DP_TUBING,AVG_CHOKE_SIZE_P,AVG_WHP_P,AVG_WHT_P,BORE_OIL_VOL,BORE_WAT_VOL,...,water_cumulative,oil_acceleration,oil_trend_strength,water_trend_strength,oil_vs_trend,water_vs_trend,oil_volatility_index,oil_momentum_30d,oil_roc_30d,oil_zscore_30
0,2014-06-06,24.0,220.749795,107.986806,157.674225,47.202699,63.075569,56.534234,690.38,92.18,...,1252.95,42.60,-53.398714,36.586429,0.954443,2.207111,0.101595,-234.93,-0.253893,-0.251521
1,2014-06-07,24.0,220.640141,107.994109,157.278305,47.194107,63.361836,56.615421,694.30,92.07,...,1345.02,-9.17,-49.170048,35.386000,0.969673,2.053575,0.099854,-219.56,-0.240256,-0.172277
2,2014-06-08,24.0,220.504050,108.002713,157.184594,47.152024,63.319457,52.051159,690.70,91.25,...,1436.27,-7.52,-71.087190,33.940048,0.942106,1.905979,0.097347,513.90,2.906674,-0.568233
3,2014-06-09,19.5,221.450514,107.757260,159.113239,38.978911,62.337275,53.559224,238.45,28.15,...,1464.42,-448.65,-79.188238,30.421714,0.336025,0.576679,0.208615,-705.74,-0.747455,-4.317439
4,2014-06-10,24.0,220.851692,107.990754,158.074501,46.512532,62.777191,54.486153,647.11,84.88,...,1549.30,860.91,-83.024429,28.189524,0.919526,1.643581,0.208276,-176.30,-0.214110,-0.526682
5,2014-06-11,24.0,218.414712,108.065274,156.685164,47.815208,61.729549,54.295347,745.92,94.23,...,1643.53,-309.85,-76.666190,27.119952,1.061093,1.720017,0.210288,-23.09,-0.030026,0.400897
6,2014-06-12,24.0,215.496859,108.139361,155.559215,48.727774,59.937644,61.561027,804.85,101.88,...,1745.41,-39.88,-60.578333,25.339667,1.141456,1.751107,0.216511,64.05,0.086461,0.918978
7,2014-06-13,24.0,214.823228,108.135199,154.685158,48.263917,60.138070,62.327179,818.89,56.75,...,1802.16,-44.89,-45.741429,18.386571,1.155596,0.944700,0.222879,105.65,0.148127,0.997787
8,2014-06-14,24.0,215.832541,108.104783,155.597785,48.005914,60.234755,59.584407,781.21,54.32,...,1856.48,-51.72,-34.571714,11.183048,1.100488,0.877790,0.225316,37.38,0.050253,0.641931
9,2014-06-15,24.0,214.877321,108.113804,154.957708,48.040325,59.919612,60.774161,786.16,54.55,...,1911.03,42.63,-22.603571,4.121857,1.104863,0.856344,0.227629,50.07,0.068022,0.666773


### Célula 2: Configuração do modelo local e estado do agente

In [6]:
import os
from typing import Any, Dict, List

import ollama
from typing_extensions import TypedDict


# -----------------------------------------------------------------------------
# BLOCO 1 - Configuração de ambiente
# -----------------------------------------------------------------------------
# Estas funções leem variáveis de ambiente de forma segura.
# Se a variável não existir, o notebook usa um valor padrão.
def read_env_text(name: str, default: str) -> str:
    raw_value = os.environ.get(name)
    if raw_value is None:
        return default

    cleaned_value = str(raw_value).strip()
    if not cleaned_value:
        return default

    return cleaned_value


def read_env_float(name: str, default: float) -> float:
    raw_value = os.environ.get(name)
    if raw_value is None:
        return default

    cleaned_value = str(raw_value).strip()
    if not cleaned_value:
        return default

    return float(cleaned_value)


def read_env_int(name: str, default: int) -> int:
    raw_value = os.environ.get(name)
    if raw_value is None:
        return default

    cleaned_value = str(raw_value).strip()
    if not cleaned_value:
        return default

    return int(cleaned_value)


# Parâmetros principais do notebook.
# Eles definem qual modelo local gera SQL e como o runtime local será acessado.
LOCAL_SQL_MODEL = read_env_text("LOCAL_SQL_MODEL", "qwen2.5-coder:7b-instruct")
OLLAMA_HOST = read_env_text("OLLAMA_HOST", "http://127.0.0.1:11434")
OLLAMA_TIMEOUT_SECONDS = read_env_float("OLLAMA_TIMEOUT_SECONDS", 180.0)
LOCAL_SQL_NUM_PREDICT = read_env_int("LOCAL_SQL_NUM_PREDICT", 80)
TEST_QUESTIONS_LIMIT = read_env_int("TEST_QUESTIONS_LIMIT", 0)
REMOTE_STAGE_STATUS = (
    "Etapa remota desabilitada neste notebook para evitar consumo de créditos."
)


# -----------------------------------------------------------------------------
# BLOCO 2 - Estado do agente
# -----------------------------------------------------------------------------
# Em LangGraph, o estado é o "pacote de informações" que circula entre os nós.
# Cada nó lê o que precisa e devolve novas chaves/valores para o fluxo.
class AgentState(TypedDict):
    question: str
    generated_sql: str
    error_message: str
    retry_count: int
    query_result: str
    query_column_context: str
    sql_generation_time: float
    sql_execution_time: float
    remote_response_time: float
    local_prompt_chars: int
    remote_prompt_chars: int
    final_response: str


# -----------------------------------------------------------------------------
# BLOCO 3 - Utilitários de apoio
# -----------------------------------------------------------------------------
# safe_str evita problemas de encoding ao montar prompts, logs e relatórios.
def safe_str(text: str | None) -> str:
    if text is None:
        return ""

    return str(text).encode("utf-8", errors="ignore").decode("utf-8")


def estimate_message_chars(messages: List[Dict[str, str]]) -> int:
    total_chars = 0

    for message in messages:
        # Medimos o tamanho aproximado do prompt local para análise de custo.
        total_chars += len(safe_str(message.get("role", "")))
        total_chars += len(safe_str(message.get("content", "")))

    return total_chars


def log_progress(message: str) -> None:
    # flush=True exibe o log imediatamente.
    print(safe_str(message), flush=True)


# Traduzimos erros técnicos do Ollama para mensagens mais legíveis.
def format_local_error(exc: Exception) -> str:
    if isinstance(exc, ollama.RequestError):
        return (
            f"Ollama indisponível em {OLLAMA_HOST}. "
            f"Inicie com `ollama serve`. Detalhe: {safe_str(str(exc))}"
        )

    if isinstance(exc, ollama.ResponseError):
        if getattr(exc, "status_code", None) == 404:
            return (
                f"Modelo local '{LOCAL_SQL_MODEL}' não encontrado. "
                f"Baixe com `ollama pull {LOCAL_SQL_MODEL}`."
            )

        return f"ResponseError(status={exc.status_code}): {safe_str(str(exc))}"

    return f"{type(exc).__name__}: {safe_str(str(exc))}"


# A resposta do modelo pode vir em mais de um formato.
# Esta função normaliza tudo para uma string única.
def extract_text_content(content: Any) -> str:
    if isinstance(content, str):
        return safe_str(content)

    if not isinstance(content, list):
        return safe_str(content)

    text_parts: List[str] = []

    for item in content:
        if isinstance(item, str):
            text_parts.append(safe_str(item))
            continue

        if not isinstance(item, dict):
            continue

        if item.get("type") != "text":
            continue

        text_value = item.get("text", "")
        if text_value:
            text_parts.append(safe_str(text_value))

    return "\n".join(part for part in text_parts if part).strip()


# -----------------------------------------------------------------------------
# BLOCO 4 - Prompt-base do modelo local
# -----------------------------------------------------------------------------
# Este prompt de sistema define o papel do modelo e suas restrições.
# Em outras palavras: é aqui que ensinamos o comportamento esperado do LLM.
LOCAL_SQL_SYSTEM_PROMPT = """
Você é um engenheiro de software especialista em banco de dados, com especialidade em SQLite.
Sua função neste fluxo é traduzir perguntas em linguagem natural para consultas SQL corretas, seguras e compatíveis com SQLite.
Use apenas o esquema fornecido e retorne somente SQL puro quando solicitado.
Jamais retorne nenhum tipo de formatação, como por exemplo markdown.
Padrões obrigatórios de execução:
- use apenas SELECT;
- prefira ORDER BY ... DESC/ASC LIMIT 1 para perguntas de máximo ou mínimo quando a resposta exigir a data associada;
- evite SELECT com MAX/MIN junto de colunas não agregadas sem ORDER BY ou subquery explícita;
- sempre use aliases legíveis para agregações;
- para colunas base retornadas diretamente, preserve exatamente o nome original sem alias;
- use alias apenas para agregações reais ou expressões calculadas;
- se a pergunta pedir média, soma, máximo, mínimo ou acumulado, nomeie a coluna de saída de forma clara;
- se a pergunta for temporal, preserve DATEPRD no resultado;
- nunca invente colunas, unidades, entidades operacionais ou lógica fora do esquema e do dicionário.
""".strip()


# Em modelos chat, enviamos uma sequência de mensagens.
# Aqui usamos o padrão clássico "system + user".
def build_local_sql_messages(prompt: str) -> List[Dict[str, str]]:
    return [
        {"role": "system", "content": LOCAL_SQL_SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]


# -----------------------------------------------------------------------------
# BLOCO 5 - Cliente Ollama
# -----------------------------------------------------------------------------
# Este cliente conversa com o servidor local do Ollama.
ollama_client = ollama.Client(host=OLLAMA_HOST, timeout=OLLAMA_TIMEOUT_SECONDS)

# Guardamos em memória se o modelo já foi validado.
# Isso evita verificações repetidas a cada pergunta.
LOCAL_SQL_MODEL_READY = False


# Mostra rapidamente se o modelo local está disponível antes dos testes.
def probe_local_sql_runtime() -> str:
    try:
        ollama_client.show(LOCAL_SQL_MODEL)
        return f"[LOCAL SQL] Modelo disponível no Ollama: {LOCAL_SQL_MODEL}"
    except Exception as exc:
        return f"[LOCAL SQL] {format_local_error(exc)}"


# Garante que o modelo exista antes de pedir qualquer geração de SQL.
def ensure_local_sql_model_ready() -> None:
    global LOCAL_SQL_MODEL_READY

    if LOCAL_SQL_MODEL_READY:
        return

    try:
        ollama_client.show(LOCAL_SQL_MODEL)
        LOCAL_SQL_MODEL_READY = True
    except Exception as exc:
        raise RuntimeError(format_local_error(exc)) from exc


# Esta é a chamada real ao LLM local.
# Ela envia as mensagens para o Ollama e devolve o texto gerado.
def invoke_local_sql_model(prompt: str) -> str:
    ensure_local_sql_model_ready()

    messages = build_local_sql_messages(prompt)
    response = ollama_client.chat(
        model=LOCAL_SQL_MODEL,
        messages=messages,
        options={
            # Temperature zero é importante aqui porque queremos SQL estável,
            # e não respostas criativas.
            "temperature": 0,
            "num_predict": LOCAL_SQL_NUM_PREDICT,
            "num_ctx": 8192,
        },
        # keep_alive deixa o modelo aquecido por um tempo, reduzindo latência
        # nas próximas perguntas.
        keep_alive="30m",
    )

    content = extract_text_content(response.message.content)
    if not content:
        raise ValueError("O modelo local retornou conteúdo vazio.")

    return content


# Diagnóstico inicial do notebook.
print(probe_local_sql_runtime())
print(f"[REMOTE RESPONSE] {REMOTE_STAGE_STATUS}")


[LOCAL SQL] Modelo disponível no Ollama: qwen2.5-coder:7b-instruct
[REMOTE RESPONSE] Etapa remota desabilitada neste notebook para evitar consumo de créditos.


### Célula 3: Nós do fluxo e compilação do LangGraph em modo sem remoto

In [7]:
from textwrap import dedent

from langgraph.graph import END, START, StateGraph


# -----------------------------------------------------------------------------
# BLOCO 1 - Regras gerais do fluxo
# -----------------------------------------------------------------------------
# MAX_SQL_RETRIES controla quantas vezes o agente pode tentar corrigir um SQL.
# PROHIBITED_KEYWORDS é uma barreira simples para impedir comandos destrutivos.
MAX_SQL_RETRIES = 3
PROHIBITED_KEYWORDS = {
    "DROP",
    "DELETE",
    "INSERT",
    "UPDATE",
    "ALTER",
    "CREATE",
    "TRUNCATE",
    "EXECUTE",
}


# Se o SQL anterior falhou, devolvemos esse contexto para o LLM tentar corrigir.
def build_retry_context(error_message: str) -> str:
    if not error_message:
        return ""

    return (
        "\nATENÇÃO: sua tentativa anterior falhou com o erro: "
        f"{error_message}. Corrija a sintaxe."
    )


# Este prompt é montado para cada pergunta.
# Aqui juntamos:
# - pergunta do usuário;
# - schema real do banco;
# - contexto semântico enxuto;
# - regras explícitas de geração de SQL.
def build_sql_prompt(question: str, error_message: str) -> str:
    retry_context = build_retry_context(error_message)
    prompt = f"""
    Sua tarefa é gerar SQL SQLite para uma série temporal real de produção offshore do projeto Volve, no Mar do Norte da Noruega.

    Tabela disponível: {TABLE_NAME}

    Esquema:
    {schema_text}

    Contexto semantico enxuto:
    {build_local_sql_context(question)}

    Regras:
    1. Retorne somente SQL puro, sem markdown.
    2. Use apenas SELECT.
    3. Use exatamente os nomes das colunas disponíveis no esquema.
    4. Use LIMIT em vez de TOP.
    5. Quando a pergunta pedir máximo, mínimo, média, tendência ou ranking temporal, retorne também as colunas de apoio necessárias para interpretar o resultado.
    6. Se a pergunta mencionar data ou depender de contexto temporal, inclua a coluna DATEPRD no resultado quando isso for necessário.
    7. Para máximo/mínimo com data associada, prefira ORDER BY com LIMIT 1 em vez de MAX/MIN com coluna não agregada solta.
    8. Preserve o nome original de colunas base retornadas diretamente, sem alias. Exemplos corretos: SELECT DATEPRD, BORE_OIL_VOL ... ; SELECT DATEPRD, AVG_DOWNHOLE_PRESSURE ...
    9. Use alias apenas para agregações reais ou expressões calculadas. Exemplos corretos: AVG(ON_STREAM_HRS) AS avg_on_stream_hrs, MAX(water_cumulative) AS max_water_cumulative.
    10. Nunca invente colunas, tabelas, datas, unidades ou métricas que não existam no esquema.
    {retry_context}

    Pergunta: {question}
    SQL:
    """
    return dedent(prompt).strip()


# O modelo às vezes devolve SQL com markdown ou ";" no final.
# Este helper limpa esse ruído antes da execução.
def clean_generated_sql(raw_sql: str) -> str:
    cleaned_sql = safe_str(raw_sql)

    for token in ["```sql", "```", ";"]:
        cleaned_sql = cleaned_sql.replace(token, "")

    cleaned_sql = cleaned_sql.strip()
    return normalize_generated_sql(cleaned_sql)


# Mesmo que o prompt diga "use apenas SELECT", ainda validamos isso em código.
# Esse tipo de dupla proteção é comum em pipelines com LLM.
def is_sql_write_command(sql: str) -> bool:
    upper_sql = safe_str(sql).upper()

    for keyword in PROHIBITED_KEYWORDS:
        if keyword in upper_sql:
            return True

    return False


# Estes dois helpers padronizam o retorno do nó de geração de SQL.
# Eles ajudam a manter o "state" consistente.
def build_sql_generation_success(
    state: AgentState,
    generated_sql: str,
    elapsed: float,
    local_prompt_chars: int,
) -> Dict[str, Any]:
    return {
        "generated_sql": safe_str(generated_sql),
        "retry_count": state.get("retry_count", 0) + 1,
        "sql_generation_time": state.get("sql_generation_time", 0.0) + elapsed,
        "local_prompt_chars": state.get("local_prompt_chars", 0) + local_prompt_chars,
    }


def build_sql_generation_error(
    state: AgentState,
    error_message: str,
    elapsed: float,
    local_prompt_chars: int,
) -> Dict[str, Any]:
    return {
        "error_message": error_message,
        "retry_count": state.get("retry_count", 0) + 1,
        "sql_generation_time": state.get("sql_generation_time", 0.0) + elapsed,
        "local_prompt_chars": state.get("local_prompt_chars", 0) + local_prompt_chars,
    }


# -----------------------------------------------------------------------------
# BLOCO 2 - Nó 1 do grafo: gerar SQL
# -----------------------------------------------------------------------------
# Esse nó recebe a pergunta e tenta devolvê-la como SQL.
# Primeiro ele tenta um fast path determinístico.
# Se não conseguir, ele chama o modelo local.
def generate_sql_node(state: AgentState) -> Dict[str, Any]:
    question = safe_str(state["question"])
    retry_count = state.get("retry_count", 0)

    # Fast path = regras manuais simples. É útil para perguntas muito diretas
    # e também reduz custo/latência.
    fast_path_sql = try_build_rule_based_sql(question)

    if fast_path_sql:
        log_progress(f"[SQL][FASTPATH] pergunta={question} sql={safe_str(fast_path_sql)}")
        return {
            "generated_sql": safe_str(fast_path_sql),
            "retry_count": retry_count + 1,
            "sql_generation_time": state.get("sql_generation_time", 0.0),
            "local_prompt_chars": state.get("local_prompt_chars", 0),
        }

    error_message = safe_str(state.get("error_message", ""))
    prompt = build_sql_prompt(question, error_message)
    local_messages = build_local_sql_messages(prompt)
    local_prompt_chars = estimate_message_chars(local_messages)
    attempt_number = retry_count + 1

    log_progress(f"[SQL][INICIO] tentativa={attempt_number} pergunta={question}")
    sql_generation_start_time = time.time()

    try:
        raw_sql = invoke_local_sql_model(prompt)
        clean_sql = clean_generated_sql(raw_sql)

        # Se a saída do modelo veio "quase certa", normalizamos antes de rodar.
        if clean_sql != raw_sql:
            log_progress(
                f"[SQL][NORMALIZADO] antes={safe_str(raw_sql)} | depois={safe_str(clean_sql)}"
            )

        elapsed = time.time() - sql_generation_start_time
        log_progress(f"[SQL][OK] tentativa={attempt_number} tempo={elapsed:.2f}s")
        return build_sql_generation_success(state, clean_sql, elapsed, local_prompt_chars)
    except Exception as exc:
        elapsed = time.time() - sql_generation_start_time
        detail = safe_str(str(exc))
        log_progress(
            f"[SQL][ERRO] tentativa={attempt_number} tempo={elapsed:.2f}s detalhe={detail}"
        )
        return build_sql_generation_error(
            state,
            f"Erro no modelo local de SQL: {detail}",
            elapsed,
            local_prompt_chars,
        )


# Resposta padrão para casos em que a execução não deve continuar.
def build_empty_execution_response(error_message: str) -> Dict[str, Any]:
    return {
        "error_message": error_message,
        "query_result": "",
        "query_column_context": "",
    }


# -----------------------------------------------------------------------------
# BLOCO 3 - Nó 2 do grafo: executar SQL
# -----------------------------------------------------------------------------
# Aqui já não estamos mais conversando com LLM.
# Este nó apenas valida o SQL e roda a consulta no SQLite.
def execute_sql_node(state: AgentState) -> Dict[str, Any]:
    error_message = safe_str(state.get("error_message", ""))
    if "Erro no modelo local de SQL" in error_message:
        return {"query_result": ""}

    generated_sql = safe_str(state.get("generated_sql", "")).strip()
    if not generated_sql:
        return build_empty_execution_response("Nenhum SQL válido foi gerado.")

    if is_sql_write_command(generated_sql):
        return build_empty_execution_response(
            "Bloqueio de segurança: comando de escrita proibido."
        )

    # O log inclui a pergunta para facilitar diagnóstico quando houver
    # múltiplas execuções em sequência.
    log_progress(
        f"[SQLITE][INICIO] executando SQL para pergunta={safe_str(state['question'])}"
    )
    sql_execution_start_time = time.time()

    try:
        # pd.read_sql executa a consulta e já devolve um DataFrame.
        df = pd.read_sql(generated_sql, conn)
        query_columns = [safe_str(str(column)) for column in df.columns.tolist()]
        query_result = safe_str(df.to_string(index=False))
        query_column_context = safe_str(
            build_query_column_context(generated_sql, query_columns)
        )
        elapsed = time.time() - sql_execution_start_time

        log_progress(
            f"[SQLITE][OK] linhas={len(df)} colunas={len(query_columns)} tempo={elapsed:.2f}s"
        )
        return {
            "query_result": query_result,
            "query_column_context": query_column_context,
            "error_message": "",
            "sql_execution_time": state.get("sql_execution_time", 0.0) + elapsed,
        }
    except Exception as exc:
        elapsed = time.time() - sql_execution_start_time
        detail = safe_str(str(exc))
        log_progress(f"[SQLITE][ERRO] tempo={elapsed:.2f}s detalhe={detail}")
        return {
            "error_message": detail,
            "query_result": "",
            "query_column_context": "",
            "sql_execution_time": state.get("sql_execution_time", 0.0) + elapsed,
        }


# -----------------------------------------------------------------------------
# BLOCO 4 - Nó 3 do grafo: resposta final local de teste
# -----------------------------------------------------------------------------
# Como a etapa remota está desligada, usamos uma resposta local explicativa.
# Isso é ótimo para debug porque expõe metadados e resultado bruto.
def build_local_test_response(state: AgentState) -> str:
    response_lines = [
        "[MODO TESTE SEM REMOTO]",
        "A etapa de resposta final por modelo remoto foi desabilitada neste notebook para evitar consumo de créditos.",
        f"Pergunta: {safe_str(state['question'])}",
    ]

    query_column_context = safe_str(state.get("query_column_context", "")).strip()
    query_result = safe_str(state.get("query_result", "")).strip()

    if query_column_context:
        response_lines.append("Metadados das colunas retornadas:\n" + query_column_context)

    if query_result:
        response_lines.append("Resultado SQL bruto:\n" + query_result)

    response_lines.append(
        "Use o SQL gerado, os metadados e o resultado bruto para validar a etapa local durante o refactor."
    )
    return "\n\n".join(response_lines)


# Este nó normalmente chamaria o modelo remoto para transformar o resultado
# técnico em linguagem mais natural. Neste notebook ele está em modo de teste.
def respond_node(state: AgentState) -> Dict[str, Any]:
    error_message = safe_str(state.get("error_message", ""))
    if error_message:
        return {
            "final_response": (
                "Não foi possível responder devido a um erro persistente: "
                f"{error_message}"
            )
        }

    log_progress(
        f"[REMOTE][SKIP] pergunta={safe_str(state['question'])} resposta remota desabilitada para testes"
    )
    final_response = build_local_test_response(state)
    return {
        "final_response": safe_str(final_response.strip()),
        "remote_response_time": state.get("remote_response_time", 0.0),
        "remote_prompt_chars": state.get("remote_prompt_chars", 0),
    }


# Esta função decide o próximo passo do grafo.
# Se houve erro e ainda há tentativas disponíveis, voltamos ao nó de geração.
# Caso contrário, seguimos para a resposta final.
def should_retry_or_respond(state: AgentState) -> str:
    has_error = bool(state.get("error_message"))
    retry_count = state.get("retry_count", 0)

    if has_error and retry_count < MAX_SQL_RETRIES:
        return "generate_sql"

    return "respond"


# -----------------------------------------------------------------------------
# BLOCO 5 - Montagem do LangGraph
# -----------------------------------------------------------------------------
# Aqui conectamos os nós em sequência:
# START -> generate_sql -> execute_sql -> respond -> END
#
# O detalhe importante é a borda condicional após execute_sql:
# dependendo do estado, o fluxo repete a geração ou encerra na resposta.
def build_workflow_app():
    workflow = StateGraph(AgentState)

    # Registro nominal dos nós do agente.
    workflow.add_node("generate_sql", generate_sql_node)
    workflow.add_node("execute_sql", execute_sql_node)
    workflow.add_node("respond", respond_node)

    # Fluxo principal.
    workflow.add_edge(START, "generate_sql")
    workflow.add_edge("generate_sql", "execute_sql")
    workflow.add_conditional_edges(
        "execute_sql",
        should_retry_or_respond,
        {"generate_sql": "generate_sql", "respond": "respond"},
    )
    workflow.add_edge("respond", END)
    return workflow.compile()


# app é o grafo já compilado e pronto para ser invocado.
app = build_workflow_app()
print(
    "[DIAGNÓSTICO] Grafo compilado: SQL local + resposta final local de teste (remoto desabilitado)."
)


[DIAGNÓSTICO] Grafo compilado: SQL local + resposta final local de teste (remoto desabilitado).


### Célula 4: Teste de estresse e geração do relatório

In [8]:
# -----------------------------------------------------------------------------
# BLOCO 1 - Perguntas de teste
# -----------------------------------------------------------------------------
# Este notebook executa uma pequena bateria de perguntas para validar o fluxo.
def get_test_questions() -> List[str]:
    questions = [
        "Em qual data ocorreu o maior BORE_OIL_VOL e qual foi o valor?",
        "Em qual data ocorreu o maior BORE_WAT_VOL e qual foi o valor?",
        "Qual foi a média de ON_STREAM_HRS?",
        "Em qual data ocorreu a maior AVG_DOWNHOLE_PRESSURE e qual foi o valor?",
        "Qual foi o maior valor de water_cumulative e em qual data ocorreu?",
        "Qual foi o maior valor de oil_roll_30 e em qual data ocorreu?",
    ]

    if TEST_QUESTIONS_LIMIT > 0:
        return questions[:TEST_QUESTIONS_LIMIT]

    return questions


# Define onde o relatório final será salvo.
def resolve_report_path() -> str:
    report_dir = os.path.join(os.getcwd(), "notebooks")
    report_name = "relatorio_08_refactor_qualidade_do_codigo.md"

    if os.path.isdir(report_dir):
        return os.path.join(report_dir, report_name)

    return os.path.abspath(report_name)


# Estado inicial = ponto de partida do agente para uma única pergunta.
# Aqui zeramos contadores, tempos e campos que serão preenchidos depois.
def build_initial_state(question: str) -> AgentState:
    return {
        "question": safe_str(question),
        "generated_sql": "",
        "error_message": "",
        "retry_count": 0,
        "query_result": "",
        "query_column_context": "",
        "sql_generation_time": 0.0,
        "sql_execution_time": 0.0,
        "remote_response_time": 0.0,
        "local_prompt_chars": 0,
        "remote_prompt_chars": 0,
        "final_response": "",
    }


# Executa uma única pergunta de ponta a ponta:
# 1. monta o estado inicial;
# 2. chama o grafo;
# 3. mede o tempo;
# 4. devolve um dicionário pronto para o relatório.
def run_single_test(index: int, total: int, question: str) -> Dict[str, Any]:
    print(f"[TESTE {index}/{total}] {question}", flush=True)

    initial_state = build_initial_state(question)
    test_start_time = time.time()

    # recursion_limit protege contra loops acidentais no grafo.
    output = app.invoke(initial_state, {"recursion_limit": 15})
    elapsed = time.time() - test_start_time

    print(f"[TESTE {index}/{total}][FIM] tempo_total={elapsed:.2f}s", flush=True)

    return {
        "index": index,
        "question": question,
        "elapsed": elapsed,
        "total_time": elapsed,
        "retry_count": max(output.get("retry_count", 0) - 1, 0),
        "generated_sql": output.get("generated_sql", ""),
        "query_result": output.get("query_result", ""),
        "error_message": output.get("error_message", ""),
        "sql_generation_time": output.get("sql_generation_time", 0.0),
        "sql_execution_time": output.get("sql_execution_time", 0.0),
        "remote_response_time": output.get("remote_response_time", 0.0),
        "local_prompt_chars": output.get("local_prompt_chars", 0),
        "remote_prompt_chars": output.get("remote_prompt_chars", 0),
        "final_response": output.get("final_response", ""),
    }


# Este helper transforma o resultado bruto de um caso em um bloco Markdown.
def build_report_case_block(result: Dict[str, Any]) -> List[str]:
    case_index = result["index"]
    question_text = safe_str(result["question"])
    total_time_text = f"{result['total_time']:.2f}"
    sql_generation_time_text = f"{result['sql_generation_time']:.2f}"
    sql_execution_time_text = f"{result['sql_execution_time']:.2f}"
    remote_response_time_text = f"{result['remote_response_time']:.2f}"
    local_prompt_chars_text = str(result["local_prompt_chars"])
    remote_prompt_chars_text = str(result["remote_prompt_chars"])
    retry_count = result["retry_count"]
    status_text = safe_str(result["error_message"]) or "Sem erros"
    sql_text = safe_str(result["generated_sql"])
    query_result_text = safe_str(result["query_result"])
    final_response_text = safe_str(result["final_response"])

    lines = [
        f"### Caso de Teste {case_index}: {question_text}",
        f"- Tempo Total: {total_time_text} segundos",
        f"- Tempo de Geração do SQL: {sql_generation_time_text} segundos",
        f"- Tempo de Execução SQL: {sql_execution_time_text} segundos",
        f"- Tempo de Resposta Remota: {remote_response_time_text} segundos (etapa desabilitada)",
        f"- Tamanho do Prompt Local Enviado: {local_prompt_chars_text} caracteres",
        f"- Tamanho do Prompt Remoto Enviado: {remote_prompt_chars_text} caracteres",
        f"- Tentativas de Correção (Retries): {retry_count}",
        f"- Status: {status_text}",
        "",
        "```sql",
        f"{sql_text}",
        "```",
        "",
        "```text",
        f"{query_result_text}",
        "```",
        "",
        f"> {final_response_text}",
        "",
        "---",
        "",
    ]
    return lines


# -----------------------------------------------------------------------------
# BLOCO 2 - Escrita do relatório
# -----------------------------------------------------------------------------
# O relatório documenta:
# - configuração da execução;
# - dicionário de dados;
# - resultados caso a caso;
# - sumário final de performance.
def write_report(
    report_path: str,
    results: List[Dict[str, Any]],
    total_duration: float,
    overall_status: str,
) -> None:
    success_count = sum(1 for result in results if not result["error_message"])
    failed_count = len(results) - success_count
    average_total_time = total_duration / len(results)
    average_sql_generation_time = (
        sum(result["sql_generation_time"] for result in results) / len(results)
    )
    average_sql_execution_time = (
        sum(result["sql_execution_time"] for result in results) / len(results)
    )
    average_remote_time = (
        sum(result["remote_response_time"] for result in results) / len(results)
    )
    average_local_prompt_chars = (
        sum(result["local_prompt_chars"] for result in results) / len(results)
    )
    average_remote_prompt_chars = (
        sum(result["remote_prompt_chars"] for result in results) / len(results)
    )

    # Montamos o relatório como lista de linhas e, no final, juntamos tudo.
    # Isso deixa a geração de Markdown mais simples de manter.
    report_lines = [
        "# Relatório do Notebook 08 - Refactor e Qualidade do Código",
        "",
        f"**Data da Execução:** {time.strftime('%Y-%m-%d %H:%M:%S')}",
        "",
        f"**Fonte de Dados SQLite:** {DB_NAME}",
        f"**Banco SQLite:** {DB_NAME}",
        f"**Tabela SQLite:** {TABLE_NAME}",
        f"**Shape do DataFrame Lido do SQLite:** {source_df.shape}",
        "",
        f"**Modelo Local (SQL):** {LOCAL_SQL_MODEL} via Ollama",
        "**Etapa Remota (Resposta Final):** desabilitada neste notebook para evitar consumo de créditos",
        "",
        "## 1. Fonte dos Dados",
        "",
        "```text",
        DATA_SOURCE_TEXT,
        "```",
        "",
        "## 2. Dicionário de Dados Utilizado",
        "",
        "```text",
        DATA_DICTIONARY,
        "```",
        "",
        "## 3. Histórico de Execuções e Respostas Técnicas",
        "",
    ]

    for result in results:
        report_lines.extend(build_report_case_block(result))

    report_lines.extend(
        [
            "## 4. Sumário Executivo de Performance",
            "",
            f"- Total de Perguntas Submetidas: {len(results)}",
            f"- Casos com sucesso: {success_count}",
            f"- Casos com falha: {failed_count}",
            f"- Tempo Total de Varredura: {total_duration:.2f} segundos",
            f"- Média de Tempo por Requisição: {average_total_time:.2f} segundos",
            f"- Média de Tempo de Geração do SQL: {average_sql_generation_time:.2f} segundos",
            f"- Média de Tempo de Execução SQL: {average_sql_execution_time:.2f} segundos",
            f"- Média de Tempo de Resposta Remota: {average_remote_time:.2f} segundos (etapa desabilitada)",
            f"- Média de Tamanho do Prompt Local: {average_local_prompt_chars:.2f} caracteres",
            f"- Média de Tamanho do Prompt Remoto: {average_remote_prompt_chars:.2f} caracteres",
            f"- Status Geral do Sistema: {overall_status}",
        ]
    )

    report_content = "\n".join(report_lines) + "\n"
    with open(report_path, "w", encoding="utf-8") as report_file:
        report_file.write(report_content)


# -----------------------------------------------------------------------------
# BLOCO 3 - Orquestração final do teste
# -----------------------------------------------------------------------------
# A partir daqui o notebook efetivamente executa a bateria de testes.
test_questions = get_test_questions()
REPORT_PATH = resolve_report_path()
results: List[Dict[str, Any]] = []
total_start_time = time.time()

print("=" * 80, flush=True)
print(f"INICIANDO TESTE DE ESTRESSE COM {len(test_questions)} PERGUNTAS", flush=True)
print("=" * 80, flush=True)

for index, question in enumerate(test_questions, start=1):
    result = run_single_test(index, len(test_questions), question)
    results.append(result)

# Consolidamos o resultado global ao final da varredura.
total_duration = time.time() - total_start_time
success_count = sum(1 for result in results if not result["error_message"])
failed_count = len(results) - success_count
overall_status = "Concluído com sucesso"

if failed_count > 0:
    overall_status = f"Concluído com falhas ({failed_count}/{len(results)})"

# O relatório é o artefato persistente desta execução.
write_report(REPORT_PATH, results, total_duration, overall_status)

# Logs finais de encerramento.
print("=" * 80, flush=True)
print(overall_status, flush=True)
print(f"Relatório salvo em: {REPORT_PATH}", flush=True)
print("=" * 80, flush=True)


INICIANDO TESTE DE ESTRESSE COM 6 PERGUNTAS
[TESTE 1/6] Em qual data ocorreu o maior BORE_OIL_VOL e qual foi o valor?
[SQL][FASTPATH] pergunta=Em qual data ocorreu o maior BORE_OIL_VOL e qual foi o valor? sql=SELECT DATEPRD, BORE_OIL_VOL FROM volve_ml_ready ORDER BY BORE_OIL_VOL DESC LIMIT 1
[SQLITE][INICIO] executando SQL para pergunta=Em qual data ocorreu o maior BORE_OIL_VOL e qual foi o valor?
[SQLITE][OK] linhas=1 colunas=2 tempo=0.00s
[REMOTE][SKIP] pergunta=Em qual data ocorreu o maior BORE_OIL_VOL e qual foi o valor? resposta remota desabilitada para testes
[TESTE 1/6][FIM] tempo_total=0.01s
[TESTE 2/6] Em qual data ocorreu o maior BORE_WAT_VOL e qual foi o valor?
[SQL][FASTPATH] pergunta=Em qual data ocorreu o maior BORE_WAT_VOL e qual foi o valor? sql=SELECT DATEPRD, BORE_WAT_VOL FROM volve_ml_ready ORDER BY BORE_WAT_VOL DESC LIMIT 1
[SQLITE][INICIO] executando SQL para pergunta=Em qual data ocorreu o maior BORE_WAT_VOL e qual foi o valor?
[SQLITE][OK] linhas=1 colunas=2 temp